In [4]:
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

import pandas as pd


from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Connect to database via CRUD Module
db = AnimalShelter()

# Load all animal records into the initial dashboard table
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId before sending records to the Dash data table
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)



#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

# Grazioso Salvare’s logo
image_filename = 'Grazioso_Salvare_Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.H3("Jade Giles Dashboard", style={'color': 'green', 'textAlign': 'center'}),
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '250px'}
        )
    ),
    html.Hr(),
    html.Div([
     dcc.RadioItems(
         id='filter-type',
         options=[
             {'label': 'Reset', 'value': 'RESET'},
             {'label': 'Water Rescue', 'value': 'WATER'},
             {'label': 'Mountain/Wilderness Rescue', 'value': 'MOUNTAIN'},
             {'label': 'Disaster/Individual Tracking', 'value': 'DISASTER'}
         ],
         value='RESET',
         inline=True
     )
    ]),
    
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         page_size=10,
                         sort_action='native',
                         filter_action='native',
                         row_selectable='single',
                         selected_rows=[0],
                         style_table={'overflowX': 'auto', 'height': '350px', 'overflowY': 'auto'},
                         

                        ),
    html.Br(),
    html.Hr(),
# This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex', 'alignItems': 'flex-start'},
        children=[
            html.Div(
                id='graph-id',
                style={'width': '45%', 'padding': '10px'}
            ),
            html.Div(
                id='map-id',
                style={'width': '55%', 'padding': '10px'}
            )
        ]
    )
])
#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    
    if filter_type == "WATER":
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Labrador|Chesapeake|Newfoundland", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    
    elif filter_type =="MOUNTAIN":
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    
    elif filter_type =="DISASTER":
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Doberman|Rottweiler|Golden Retriever|Bloodhound", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    
    else: # RESET
        query = {}
        
    filtered_df = pd.DataFrame.from_records(db.read(query))
    
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
    return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return html.Div("No graph data available.")

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return html.Div("No graph data available.")

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    breed_counts = breed_counts.head(10)

    return [
        dcc.Graph(
            figure=px.bar(
                breed_counts,
                x='breed',
                y='count',
                title='Top Breeds in Current Filter'
            )
        )
    ]
    
# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update the map based on the animal selected in the filtered data table
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if not viewData:
        return html.Div("No map data available.")
    
    dff = pd.DataFrame.from_dict(viewData)

    required_columns = ['location_lat', 'location_long', 'breed', 'name']
    if not all(col in dff.columns for col in required_columns):
        return html.Div("Location data is unavailable.")
    
    if not index:
        row = 0
    else: 
        row = index[0]

    if row >= len(dff):
        row = 0

    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']

    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[latitude, longitude], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[latitude, longitude], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


# Run the Dashboard in JupyterLab
app.run(jupyter_mode='inline', port=8054, debug=False)
